# Ingest sources -> landing volume (bronze input)
Daily task of the `daily_refresh` job. Lands raw files, one folder per source and `run_date`:

| Folder | Source | Cadence |
| --- | --- | --- |
| `reference/natural_earth/` | Natural Earth 10m admin-0 boundaries (public domain) | if missing |
| `reference/grid_cells/` | Land grid: 0.18 deg (~20 km) PE/CO/EC, 0.09 deg (~10 km) CR/PA | if missing |
| `geonames/` | GeoNames country dumps, CC-BY 4.0 | if missing |
| `enso_oni/run_date=` | NOAA CPC Oceanic Nino Index | daily |
| `gdacs/run_date=` | GDACS events, last 30 days | daily |
| `open_meteo_forecast/run_date=` | Open-Meteo: 10 daily vars, yesterday + today + 11 days | daily |

Open-Meteo free tier (non-commercial, CC-BY 4.0): <10k calls/day, 5k/hour, 600/min, 300k/month.
One location = 1 call while <=10 variables and <=14 days, so the ~8k grid fits once a day.
Horizon is +/- 11 days: the daily run keeps 13 days (1 call/point); the past 11 days are built
from stored daily runs. One-off backfill: `past_days=11, forecast_days=1` (12 days, 1 call/point)
on a day without the scheduled run. A budget guard stops runs that would exceed `daily_call_budget`.
Batches already landed for a `run_date` are skipped, so a failed run can be resumed.
Code lives next to this notebook: `ingest_lib.py` (Open-Meteo, rate limiter, writers),
`context_lib.py` (grid, GeoNames, ENSO, GDACS), `source_registry.py` (sources, licenses, tags).
Every JSONL row carries `data_source`, `source_model` and `request_mode` (here: `daily`).
Backfills run from `backfill_sources`.

In [0]:
import datetime as dt, json
import context_lib as ctx  # grid, GeoNames, ENSO, GDACS (same folder)
import ingest_lib as lib   # Open-Meteo downloader, rate limiter, writers

dbutils.widgets.text("catalog", "weather_forecast_warnings_comms")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("volume", "landing")
dbutils.widgets.text("run_date", "")          # blank = today (UTC)
dbutils.widgets.text("countries", "PER,ECU,COL,PAN,CRI")
dbutils.widgets.text("max_points", "0")       # 0 = all grid points (testing: e.g. 50)
dbutils.widgets.text("sources", "reference,enso,gdacs,forecast")
dbutils.widgets.text("past_days", "1")       # daily: yesterday; backfill: 11
dbutils.widgets.text("forecast_days", "12")   # today + 11 days ahead
dbutils.widgets.text("daily_call_budget", "9500")  # stay under the 10k/day free tier

w = dbutils.widgets.get
ROOT = f"/Volumes/{w('catalog')}/{w('schema')}/{w('volume')}"
RUN_DATE = w("run_date") or dt.datetime.now(dt.timezone.utc).date().isoformat()
COUNTRIES = [c.strip() for c in w("countries").split(",") if c.strip()]
MAX_POINTS = int(w("max_points"))
SOURCES = set(s.strip() for s in w("sources").split(","))
PAST_DAYS, FORECAST_DAYS = int(w("past_days")), int(w("forecast_days"))
DAILY_CALL_BUDGET = float(w("daily_call_budget"))
INGESTED_AT = dt.datetime.now(dt.timezone.utc).isoformat()
summary = {"run_date": RUN_DATE}
print(ROOT, RUN_DATE, COUNTRIES, MAX_POINTS, SOURCES, PAST_DAYS, FORECAST_DAYS)

In [0]:
GRID_DIR = f"{ROOT}/reference/grid_cells"
if "reference" in SOURCES:
    ctx.land_reference(spark, ROOT)  # source manifest; boundaries, GeoNames, land grid (if missing)

grid = spark.read.json(GRID_DIR).where(f"iso3 IN ({','.join(repr(c) for c in COUNTRIES)})")
display(grid.groupBy("iso3", "grid_km").count().orderBy("iso3"))

In [0]:
if "enso" in SOURCES:
    summary["enso_oni_lines"] = ctx.land_enso(ROOT, RUN_DATE)
if "gdacs" in SOURCES:
    summary["gdacs_events"] = ctx.land_gdacs(ROOT, RUN_DATE, COUNTRIES, INGESTED_AT)
print(summary)

In [0]:
if "forecast" in SOURCES:
    pts = grid.select("cell_id", "lat", "lon").orderBy("cell_id")
    pts = (pts.limit(MAX_POINTS) if MAX_POINTS else pts).collect()  # <= ~8k small rows
    summary.update(lib.land_forecast(ROOT, RUN_DATE, pts, PAST_DAYS, FORECAST_DAYS, INGESTED_AT, DAILY_CALL_BUDGET))

print(json.dumps(summary))
dbutils.notebook.exit(json.dumps(summary))